In [ ]:
import json
import pandas as pd
import re
import ast

In [7]:
FULL_DATA_PATH = "../data/output/full_dataset.json"
TAXONOMY_PATH = "../data/output/taxonomy_autogen_v3.csv"

STRIPPED_DATA_PATH = "../data/output/stripped_annotated_data.csv"
GENERATED_NOTES_PATH = "../data/input/generated_fake_data.tsv"

GENERATED_NOTES_OUTPUT_PATH = "../data/output/generated_fake_data.json"

## 1. Strip Annotated Data of Note Content & Export to CSV

In [8]:
with open(FULL_DATA_PATH, 'r', encoding='utf-8') as f:
    records = json.load(f)

len(records)

2000

In [9]:
records[1]

{'id': '685524bb-94d4-4f08-8669-0e78df744c0c',
 'text': '[Category: tenureManagement] Welfare check and tenancy audit completed I visited the tenant in his property following his request for me to come over and assess the toilet in his property. I used the opportunity to complete a tenancy audit and I observed that he is a hoarder with so many items in nearly all the rooms. The toilet seat has broken and the toilet is as dark as charcoal.',
 'needs': [{'id': '3aefb716',
   'start': 268,
   'end': 275,
   'label': 'housing_conditions_hoarding'},
  {'id': '99751da7',
   'start': 281,
   'end': 318,
   'label': 'housing_conditions_hoarding'},
  {'id': '354b3b37',
   'start': 324,
   'end': 346,
   'label': 'property_level_disrepair_damp_mould'},
  {'id': 'c8515bee',
   'start': 355,
   'end': 384,
   'label': 'cautions_unclean_unsafe_living_environment'}],
 'entities': [{'id': '2373321f',
   'start': 85,
   'end': 91,
   'label': 'Person_Role'}],
 'relations': [{'from': '3aefb716', 'to': 

In [10]:
taxonomy = pd.read_csv(TAXONOMY_PATH, index_col='cat_label')
taxonomy.head(5)

,high_level_category,category_description,values_hint,regex
cat_label,,,,
care_care_setting,Care,Care setting,"['Fostered', 'Social care']",\bfostered\b|\bfoster care\b|\bfoster placemen...
care_has_caring_responsibility,Care,Has caring responsibility,"['Formal', 'Informal']",\bformal carer\b|\bregistered carer\b|\bcarer....
care_social_care_involvement,Care,Social care involvement,"['Adult Social Care', ""Children's Social Care""]",\bsocial care\b|\bcare package\b|\bcare plan\b...
cautions_asbo_or_injunction_obtained,Cautions,ASBO or injunction obtained,"['ASBO', 'Injunction']",\bASBO\b|\binjunction\b|\bcivil injunction\b|\...
cautions_physical_abuse_or_threat_of,Cautions,Physical Abuse or Threat of,"['Physical Abuse (actual)', 'Physical Abuse (t...",\bphysical abuse\b|\bthreat of physical abuse\...


In [11]:
stripped_records = []

for r in records:
    # Extract category from note
    category = None
    category_match = re.search(r'\[Category:\s*([^\]]+)\]', r['text'])
    if category_match:
        category = category_match.group(1)
    
    # Get person entities
    entities = [e['label'] for e in r['entities']]

    # Get additional needs
    needs = [n['label'] for n in r['needs']]
    
    # Look up AN description and examples from taxonomy (filtered per example)
    taxonomy_strings = []
    for label in set(needs):
        if label in taxonomy.index:
            row = taxonomy.loc[label]
            desc = row['category_description']
            hints = row['values_hint']
            taxonomy_strings.append({"label": label, "meaning": desc, "examples/subcategories": {hints}})

    stripped_records.append({
        'id': r['id'],
        'category': category or "General",
        'need_labels': needs,
        'entity_labels': entities,
        'relation_count': len(r['relations']),
        'note_length': len(r['text'].split()),
        'need_taxonomy': taxonomy_strings,
    })
    
print("length of stripped_records:", len(stripped_records))
stripped_records[1]  

length of stripped_records: 2000


{'id': '685524bb-94d4-4f08-8669-0e78df744c0c',
 'category': 'tenureManagement',
 'need_labels': ['housing_conditions_hoarding',
  'housing_conditions_hoarding',
  'property_level_disrepair_damp_mould',
  'cautions_unclean_unsafe_living_environment'],
 'entity_labels': ['Person_Role'],
 'relation_count': 3,
 'note_length': 69,
 'need_taxonomy': [{'label': 'housing_conditions_hoarding',
   'meaning': 'Hoarding',
   'examples/subcategories': {"['Clutter Image Rating 1-3', 'Clutter Image Rating 4-6', 'Clutter Image Rating 7-9']"}},
  {'label': 'cautions_unclean_unsafe_living_environment',
   'meaning': 'Unclean / Unsafe Living Environment',
   'examples/subcategories': {"['Unclean and / or unsanitary', 'Unsafe e.g. undisposed sharps']"}},
  {'label': 'property_level_disrepair_damp_mould',
   'meaning': 'Disrepair, damp & mould',
   'examples/subcategories': {"['Disrepair', 'Damp', 'Mould', 'Leaks']"}}]}

In [12]:
stripped_df = pd.DataFrame(stripped_records)
stripped_df.to_csv(STRIPPED_DATA_PATH, index=False)

## 2. Handle Gemini Generated Notes

In [13]:
synthetic_notes = pd.read_csv(GENERATED_NOTES_PATH, sep='\t')
synthetic_notes.head()

,id,category,need_labels,entity_labels,relation_count,note_length,need_taxonomy,generated_data
0,4d1f84da-85f0-9837-33ac-bdc3ae96fcba,Rents,[],[],0,6,[],"{ ""title"": ""Discussion regarding outstanding r..."
1,685524bb-94d4-4f08-8669-0e78df744c0c,tenureManagement,"['housing_conditions_hoarding', 'housing_condi...",['Person_Role'],3,69,[{'label': 'property_level_disrepair_damp_moul...,"{ ""title"": ""Property inspection and welfare re..."
2,b7b3fb41-80b7-4d97-b8f4-0e0a7156ab79,estateManagement,"['safety_risk_firerelated_risks', 'safety_risk...","['Person_Role', 'Person_Role', 'Person_Role']",0,159,"[{'label': 'safety_risk_firerelated_risks', 'm...","{ ""title"": ""Estate management inspection and f..."
3,918721e6-ccf6-445c-8cfc-658826b245cd,repairs,['disability_requires_adapted_property'],"['Person_Role', 'Person_Role']",1,42,[{'label': 'disability_requires_adapted_proper...,"{ ""title"": ""Discussion regarding property adap..."
4,5cac962d-dfc9-72d1-175a-612d7462cd4d,Tenancy Management,"['health_mental_health', 'safety_risk_domestic...","['Person_Name', 'Person_Name', 'Person_Name', ...",6,135,[{'label': 'cautions_physical_abuse_or_threat_...,"{ ""title"": ""Tenancy management visit following..."


In [14]:
def sanitise_data(raw_string):
    # Remove json wrapper (markdown)
    cleaned = raw_string.replace('``` json', '')
    cleaned = cleaned.replace('```', '')
    # Look for double quotes inside XML tags and replace them with single quotes
    cleaned = re.sub(r'=(["\'])(.*?)\1(?=[^<>]*>)', r"='\2'", cleaned)
    
    return cleaned

def safe_parse_list(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except:
        return []

synthetic_notes['generated_data_clean'] = synthetic_notes['generated_data'].apply(sanitise_data)
synthetic_notes['need_labels_clean'] = synthetic_notes['need_labels'].apply(safe_parse_list)
synthetic_notes['entity_labels_clean'] = synthetic_notes['entity_labels'].apply(safe_parse_list)
synthetic_notes.head()

,id,category,need_labels,entity_labels,relation_count,note_length,need_taxonomy,generated_data,generated_data_clean,need_labels_clean,entity_labels_clean
0,4d1f84da-85f0-9837-33ac-bdc3ae96fcba,Rents,[],[],0,6,[],"{ ""title"": ""Discussion regarding outstanding r...","{ ""title"": ""Discussion regarding outstanding r...",[],[]
1,685524bb-94d4-4f08-8669-0e78df744c0c,tenureManagement,"['housing_conditions_hoarding', 'housing_condi...",['Person_Role'],3,69,[{'label': 'property_level_disrepair_damp_moul...,"{ ""title"": ""Property inspection and welfare re...","{ ""title"": ""Property inspection and welfare re...","[housing_conditions_hoarding, housing_conditio...",[Person_Role]
2,b7b3fb41-80b7-4d97-b8f4-0e0a7156ab79,estateManagement,"['safety_risk_firerelated_risks', 'safety_risk...","['Person_Role', 'Person_Role', 'Person_Role']",0,159,"[{'label': 'safety_risk_firerelated_risks', 'm...","{ ""title"": ""Estate management inspection and f...","{ ""title"": ""Estate management inspection and f...","[safety_risk_firerelated_risks, safety_risk_fi...","[Person_Role, Person_Role, Person_Role]"
3,918721e6-ccf6-445c-8cfc-658826b245cd,repairs,['disability_requires_adapted_property'],"['Person_Role', 'Person_Role']",1,42,[{'label': 'disability_requires_adapted_proper...,"{ ""title"": ""Discussion regarding property adap...","{ ""title"": ""Discussion regarding property adap...",[disability_requires_adapted_property],"[Person_Role, Person_Role]"
4,5cac962d-dfc9-72d1-175a-612d7462cd4d,Tenancy Management,"['health_mental_health', 'safety_risk_domestic...","['Person_Name', 'Person_Name', 'Person_Name', ...",6,135,[{'label': 'cautions_physical_abuse_or_threat_...,"{ ""title"": ""Tenancy management visit following...","{ ""title"": ""Tenancy management visit following...","[health_mental_health, safety_risk_domestic_ab...","[Person_Name, Person_Name, Person_Name, Person..."


In [15]:
# Find broken notes & fix them
def is_valid_json(x):
    try:
        json.loads(x)
        return True
    except (ValueError, TypeError):
        return False

mask = synthetic_notes['generated_data_clean'].apply(lambda x: not is_valid_json(x))
valid_predictions = synthetic_notes[~mask]

display(synthetic_notes[mask])

,id,category,need_labels,entity_labels,relation_count,note_length,need_taxonomy,generated_data,generated_data_clean,need_labels_clean,entity_labels_clean


In [16]:
def parse_synthetic_record(row):
    # Parses a row from the gemini csv & converts inline XML into character indices and maps relations

    # 1. Parse note data
    id = row['id']
    category = row['category']
    try:
        # 2. Parse the stringified JSON data from Gemini
        gen_data = json.loads(row['generated_data_clean'])
        raw_title = gen_data['title']
        raw_note_content = gen_data['note']
        relations = gen_data.get('relations', [])

        # 3. Reconstruct note string ([Category: {note_category}] {title} {content}) - Same as in gold standard construction script
        note_content = f'[Category: {category}] {raw_title} {raw_note_content}'
        
        # 4. Setup regex to capture tags: <type label='...' id='...'>text</type>
        tag_pattern = r'<(need|entity)\s+label=["\']([^"\']+)["\']\s+id=["\']([^"\']+)["\']>([\s\S]*?)</\1>'
        
        clean_text = ""
        needs = []
        entities = []
        
        last_idx = 0
        offset = 0  # Tracks characters removed by stripping XML tags
        
        # Iterate through all inline tags found in the text
        for match in re.finditer(tag_pattern, note_content):
            tag_type = match.group(1)   # 'need' or 'entity'
            label = match.group(2)      # e.g., 'health_mental_health'
            tag_id = match.group(3)     # e.g., 'n1a2b3c4'
            inner_text = match.group(4)  # e.g., 'severe depression and constant anxiety'
            
            # Calculate clean start and end positions
            clean_start = match.start() - offset
            clean_end = clean_start + len(inner_text)
            
            # Update offset tracking for subsequent tags
            offset += len(match.group(0)) - len(inner_text)
            
            # Structure the item
            item = {
                "id": tag_id,
                "start": clean_start,
                "end": clean_end,
                "label": label
            }
            
            if tag_type == 'need':
                needs.append(item)
            else:
                entities.append(item)
                
        # Reconstruct the clean text completely stripped of XML tags
        clean_text = re.sub(tag_pattern, r'\4', note_content)

        return {
            "id": id, # Fake data shares UUID with the real note - easier to compare
            "text": clean_text,
            "needs": needs,
            "entities": entities,
            "relations": relations
        }
    except Exception as e:
        print(f"Error parsing row id: {id}, error: {e}")

In [17]:
parsed_records = synthetic_notes.apply(parse_synthetic_record, axis=1) # type: ignore

  0%|          | 0/1500 [00:00<?, ?it/s]

In [18]:
parsed_records[1]

{'id': '685524bb-94d4-4f08-8669-0e78df744c0c',
 'text': '[Category: tenureManagement] Property inspection and welfare review conducted. Visited the resident to check property conditions. The flat has severe hoarding with clothes piled high everywhere which is a safety risk. It is an extremely unclean living environment with garbage covering the floors and flies everywhere. There is also a secondary hoarding situation in the back bedroom that blocks access. Also spotted heavy mould on the bathroom ceiling needing repair. Spoke to them about fixing this up soon.',
 'needs': [{'id': 'n1111111',
   'start': 150,
   'end': 182,
   'label': 'housing_conditions_hoarding'},
  {'id': 'n2222222',
   'start': 237,
   'end': 296,
   'label': 'cautions_unclean_unsafe_living_environment'},
  {'id': 'n3333333',
   'start': 345,
   'end': 383,
   'label': 'housing_conditions_hoarding'},
  {'id': 'n4444444',
   'start': 417,
   'end': 452,
   'label': 'property_level_disrepair_damp_mould'}],
 'entities

## Make sure Gemini followed instructions

In [19]:
def validate_synthetic_record(parsed, original_row):
    id = parsed['id']
    issues = []

    # Check that all need labels have been generated (at least once)
    expected_need_labels = set(safe_parse_list(original_row['need_labels']))
    actual_need_labels = set(n['label'] for n in parsed['needs'])
    missing = expected_need_labels - actual_need_labels
    if missing:
        issues.append(f"missing need labels: {missing}")

    # Check all relation IDs actually exist
    need_ids = {n['id'] for n in parsed['needs']}
    entity_ids = {e['id'] for e in parsed['entities']}
    for rel in parsed['relations']:
        if 'from' not in rel or 'to' not in rel:
            issues.append(f"relation malformed.")
            continue
        if rel['from'] not in need_ids:
            issues.append(f"relation 'from' id {rel['from']} not found in needs")
        if rel['to'] not in entity_ids:
            issues.append(f"relation 'to' id {rel['to']} not found in entities")

    return {"id": id, "valid": len(issues) == 0, "issues": issues}

In [20]:
results = [validate_synthetic_record(parsed, row) for parsed, row in zip(parsed_records, synthetic_notes.to_dict('records'))]

invalid = [r for r in results if not r['valid']]

In [21]:
print(f"{len(invalid)} / {len(results)} records have issues")
invalid

0 / 1500 records have issues


[]

## Output to JSON

In [22]:
parsed_records.to_json(GENERATED_NOTES_OUTPUT_PATH, orient='records')